In [1]:
import pandas as pd
from scipy import stats
import numpy as np
from sklearn.metrics import f1_score, mean_absolute_error, mean_squared_error
from math import sqrt

def score2label(score, res=0, size=7, m=0.49, M=1.69):
    step = (M-m) / (size)
    if score < step + m:
        return 1+res
    else:
        return score2label(score-step, res+1, size, m, M)

def sigmoid(z):
    return 1/(1 + np.exp(-z))

def quadratic_weighted_kappa(y_true, y_pred, s_min, s_max):
    """
    Calculate the Quadratic Weighted Kappa (QWK) between two raters.

    Args:
    y_true (np.array): The true labels or ratings (Rater B).
    y_pred (np.array): The predicted labels or ratings (Rater A).
    L (int): The number of possible score categories.

    Returns:
    float: The QWK score.
    """

    # Create the weights matrix
    w = np.zeros((s_max-s_min+1, s_max-s_min+1))
    for i in range(s_max-s_min+1):
        for j in range(s_max-s_min+1):
            w[i, j] = ((i - j) ** 2) / ((s_max - s_min) ** 2)

    # Calculate the O (observed) matrix
    O = np.histogram2d(y_true, y_pred, bins=s_max-s_min+1, range=[[s_min, s_max], [s_min, s_max]])[0]

    # Calculate the E (expected) matrix
    hist_true = np.histogram(y_true, bins=s_max-s_min+1, range=[s_min, s_max])[0]
    hist_pred = np.histogram(y_pred, bins=s_max-s_min+1, range=[s_min, s_max])[0]
    E = np.outer(hist_true, hist_pred) / len(y_true)

    # Compute QWK
    num = np.sum(w * O)
    denom = np.sum(w * E)
    kappa = 1 - (num / denom)

    return kappa

METRICS_ULRA_PAPER = ".759 .508 .608 .644 .711 .577 .661 .446".split()

S_MAX = 10
S_MIN = 0

results = []

scale_preds_per_set_and_model = {}
essay_ids_per_set = {}
#, "2_1", "2_2", 3, 4, 5, 6, 7, 8
for jx in [1]:
    scale_preds_per_set_and_model[jx] = {}

    df = pd.read_csv(f"{jx}/human/pred/test.csv", index_col=0)

    essay_ids_per_set[jx] = df["essay_id"].values

    y_1 = df["pred_1"]
    y_2 = df["pred_2"]

    m, M = np.min(y_1), np.max(y_1)
    scaled_1 = (S_MIN + ((pd.Series(y_1) - m) / (M - m)) * (S_MAX - S_MIN)).values

    m, M = np.min(y_2), np.max(y_2)
    scaled_2 = (S_MIN + ((pd.Series(y_2) - m) / (M - m)) * (S_MAX - S_MIN)).values

    qwk_human = np.round(quadratic_weighted_kappa(scaled_1, scaled_2, S_MIN, S_MAX), 4)

    if jx in ["2_1", "2_2"]:
        ULRA_PAPER = np.round(float(METRICS_ULRA_PAPER[2-1]), 4)
        df = pd.read_csv(f"data/essay_set_2.csv", index_col=0)
    else:
        ULRA_PAPER = np.round(float(METRICS_ULRA_PAPER[jx-1]), 4)
        df = pd.read_csv(f"data/essay_set_{jx}.csv", index_col=0)

    
    df = df[df["split"] == "test"]
    if jx == "2_1":
        df["score"] = df["domain1_score"]
    elif jx == "2_2":
        df["score"] = df["domain2_score"]


    index_to_score = dict(zip(df.index, df.score))
    essay_id_to_score = dict(zip(df.essay_id, df.score))

    results.append({
        "prompt": jx,
        "method": "ULRA (Paper)", 
        "corr.": np.nan, 
        "mae.": np.nan, 
        "qwk": ULRA_PAPER
    })

    results.append({
        "prompt": jx,
        "method": "human", 
        "corr.": np.nan, 
        "mae.": np.nan, 
        "qwk": qwk_human
    })

    for model_path in [
        "dummy/pred/random",
        "dummy/pred/length",
        "similarity/pred/jaccard",        
        "similarity/pred/cosine",

        "ulra_paper/none/pred/lf",

        "signal_clustering/none/data",

        "llm/vanilla/data",
        "llm/cot/data",

        "nllf_method/lr_Z_C/pred/ulra/wo_sf/nllf",
        "nllf_method/lr_Z_C/pred/ulra/wo_sf/ef",
        "nllf_method/lr_Z_C/pred/ulra/wo_sf/nllf_ef",

        "nllf_method/lr_Z_C/pred/signal_clustering/wo_sf/nllf",
        "nllf_method/lr_Z_C/pred/signal_clustering/wo_sf/ef",
        "nllf_method/lr_Z_C/pred/signal_clustering/wo_sf/nllf_ef",
        
        "nllf_method/lr_Z_C/pred/llm/wo_sf/nllf",
        "nllf_method/lr_Z_C/pred/llm/wo_sf/ef",
        "nllf_method/lr_Z_C/pred/llm/wo_sf/nllf_ef",
        
        "nllf_method/lr_Z_C/pred/ulra/w_sf/nllf",
        "nllf_method/lr_Z_C/pred/ulra/w_sf/ef",
        "nllf_method/lr_Z_C/pred/ulra/w_sf/nllf_ef",

        "nllf_method/lr_Z_C/pred/signal_clustering/w_sf/nllf",
        "nllf_method/lr_Z_C/pred/signal_clustering/w_sf/ef",
        "nllf_method/lr_Z_C/pred/signal_clustering/w_sf/nllf_ef",

        "nllf_method/lr_Z_C/pred/llm/w_r_sf/nllf",
        "nllf_method/lr_Z_C/pred/llm/w_r_sf/ef",
        "nllf_method/lr_Z_C/pred/llm/w_r_sf/nllf_ef",

        "bert/pred/signal_clustering/wo_sf",
        "bert/pred/llm/wo_sf",
        "bert/pred/signal_clustering/w_sf",
        "bert/pred/llm/w_sf",

        "bert/pred/ulra/wo_sf",
        "bert/pred/ulra/w_sf"

    ]:
        df_test = pd.read_csv(f"{jx}/"+model_path+f"/test.csv", index_col=0)

        if "essay_id" in df_test.columns:
            df_test["test"] = df_test["essay_id"].map(essay_id_to_score)
            
        elif "index" in df_test.columns:
            df_test["test"] = df_test["index"].map(index_to_score)

        y_true = df_test["test"].values
        y_pred = df_test["pred"].values

        m, M = np.min(y_true), np.max(y_true)
        scaled_true = (S_MIN + ((pd.Series(y_true) - m) / (M - m)) * (S_MAX - S_MIN)).values

        m, M = np.min(y_pred), np.max(y_pred)
        if "nllf_method" in model_path:
            m, M = np.percentile(y_pred, 1), np.percentile(y_pred, 99) 
        scale_preds = (S_MIN + ((pd.Series(y_pred) - m) / (M - m)) * (S_MAX - S_MIN)).values if abs(M-m) > 0 else (pd.Series(y_pred) - m)  + (S_MIN + S_MAX) / 2
        o = {
            "prompt": jx,
            "method": model_path.replace("/pred", "").replace("/data", ""), 
            "corr.": np.round(abs(stats.pearsonr(scaled_true, scale_preds)[0]), 4), 
            "mae.": np.round(mean_absolute_error(scaled_true, scale_preds), 4),
            "qwk": np.round(quadratic_weighted_kappa(scaled_true, scale_preds, S_MIN, S_MAX), 4)
        
        }
        print(o.items())
        results.append(o)

        scale_preds_per_set_and_model[jx][o["method"]] = scale_preds

    # pd.DataFrame(results)
table = pd.DataFrame(results)
prompt_2 = table[table["prompt"] == "2_1"]#.isin("2_1 2_2".split())].drop(columns="prompt").groupby("method").mean().reset_index()
prompt_2["prompt"] = "2"

new_table = pd.concat([prompt_2, table[~table["prompt"].isin("2_1 2_2".split())]], ignore_index=True)
new_table

dict_items([('prompt', 1), ('method', 'dummy/random'), ('corr.', np.float64(0.0071)), ('mae.', np.float64(2.9343)), ('qwk', np.float64(0.008))])
dict_items([('prompt', 1), ('method', 'dummy/length'), ('corr.', np.float64(0.8392)), ('mae.', np.float64(1.9031)), ('qwk', np.float64(0.5186))])
dict_items([('prompt', 1), ('method', 'similarity/jaccard'), ('corr.', np.float64(0.6779)), ('mae.', np.float64(4.2642)), ('qwk', np.float64(-0.1566))])
dict_items([('prompt', 1), ('method', 'similarity/cosine'), ('corr.', np.float64(0.0045)), ('mae.', np.float64(1.6933)), ('qwk', np.float64(-0.0036))])
dict_items([('prompt', 1), ('method', 'ulra_paper/none/lf'), ('corr.', np.float64(0.8326)), ('mae.', np.float64(1.2857)), ('qwk', np.float64(0.6741))])
dict_items([('prompt', 1), ('method', 'signal_clustering/none'), ('corr.', np.float64(0.8194)), ('mae.', np.float64(0.9699)), ('qwk', np.float64(0.7623))])
dict_items([('prompt', 1), ('method', 'llm/vanilla'), ('corr.', np.float64(0.5052)), ('mae.', np

,prompt,method,corr.,mae.,qwk
0,1,ULRA (Paper),NaN,NaN,0.7590
1,1,human,NaN,NaN,0.7481
2,1,dummy/random,0.0071,2.9343,0.0080
3,1,dummy/length,0.8392,1.9031,0.5186
4,1,similarity/jaccard,0.6779,4.2642,-0.1566
5,1,similarity/cosine,0.0045,1.6933,-0.0036
6,1,ulra_paper/none/lf,0.8326,1.2857,0.6741
7,1,signal_clustering/none,0.8194,0.9699,0.7623
8,1,llm/vanilla,0.5052,1.2881,0.4858
9,1,llm/cot,0.3423,1.7486,0.2543


In [2]:
summary = new_table.drop(columns="prompt").set_index("method")#.mean()
summary.sort_values("qwk", ascending=False)#.head(10)

,corr.,mae.,qwk
method,,,
signal_clustering/none,0.8194,0.9699,0.7623
ULRA (Paper),NaN,NaN,0.7590
human,NaN,NaN,0.7481
bert/signal_clustering/w_sf,0.7969,1.2965,0.6790
ulra_paper/none/lf,0.8326,1.2857,0.6741
bert/llm/w_sf,0.7456,1.2149,0.6685
bert/llm/wo_sf,0.7262,1.2249,0.6584
nllf_method/lr_Z_C/ulra/w_sf/nllf_ef,0.8206,1.4481,0.6254
nllf_method/lr_Z_C/llm/w_r_sf/nllf_ef,0.7185,1.3199,0.6195


In [3]:
our_model = "nllf_method/lr_Z_C"

rows = []
o = {
    "method": "Lenght", 
    "weak_signal": "None",
    "signal_filtering": "-",
    "text": summary.loc["dummy/length"]["qwk"],
    "ef": "-",
    "nllf": "-",
    "ef+nllf": "-"
}
rows.append(o)

o = {
    "method": "Jaccard Sim.", 
    "weak_signal": "None",
    "signal_filtering": "-",
    "text": summary.loc["similarity/jaccard"]["qwk"],
    "ef": "-",
    "nllf": "-",
    "ef+nllf": "-"
}
rows.append(o)

o = {
    "method": "Jaccard Sim.", 
    "weak_signal": "None",
    "signal_filtering": "-",
    "text": summary.loc["similarity/cosine"]["qwk"],
    "ef": "-",
    "nllf": "-",
    "ef+nllf": "-"
}
rows.append(o)

o = {
    "method": "ULRA", 
    "weak_signal": "LF",
    "signal_filtering": "-",
    "text": summary.loc["ulra_paper/none/lf"]["qwk"],
    "ef": "-",
    "nllf": "-",
    "ef+nllf": "-"
}
rows.append(o)

o = {
    "method": "Z-score", 
    "weak_signal": "None",
    "signal_filtering": "-",
    "text": summary.loc["signal_clustering/none"]["qwk"],
    "ef": "-",
    "nllf": "-",
    "ef+nllf": "-"
}
rows.append(o)

o = {
    "method": "LLM", 
    "weak_signal": "None",
    "signal_filtering": "-",
    "text": summary.loc["llm/vanilla"]["qwk"],
    "ef": "-",
    "nllf": "-",
    "ef+nllf": "-"
}
rows.append(o)

o = {
    "method": "LLM-CoT", 
    "weak_signal": "None",
    "signal_filtering": "-",
    "text": summary.loc["llm/cot"]["qwk"],
    "ef": "-",
    "nllf": "-",
    "ef+nllf": "-"
}
rows.append(o)

o = {
    "method": "Linear Regression", 
    "weak_signal": "Z-score",
    "signal_filtering": "xmark",
    "text": "-",
    "ef": summary.loc[f"{our_model}/signal_clustering/wo_sf/ef"]["qwk"],
    "nllf": summary.loc[f"{our_model}/signal_clustering/wo_sf/nllf"]["qwk"],
    "ef+nllf": summary.loc[f"{our_model}/signal_clustering/wo_sf/nllf_ef"]["qwk"]
}
rows.append(o)

o = {
    "method": "Linear Regression", 
    "weak_signal": "LLM-based signal",
    "signal_filtering": "xmark",
    "text": "-",
    "ef": summary.loc[f"{our_model}/llm/wo_sf/ef"]["qwk"],
    "nllf": summary.loc[f"{our_model}/llm/wo_sf/nllf"]["qwk"],
    "ef+nllf": summary.loc[f"{our_model}/llm/wo_sf/nllf_ef"]["qwk"],
}
rows.append(o)

o = {
    "method": "Linear Regression", 
    "weak_signal": "ULRA-based signal",
    "signal_filtering": "xmark",
    "text": "-",
    "ef": summary.loc[f"{our_model}/ulra/wo_sf/ef"]["qwk"],
    "nllf": summary.loc[f"{our_model}/ulra/wo_sf/nllf"]["qwk"],
    "ef+nllf": summary.loc[f"{our_model}/ulra/wo_sf/nllf_ef"]["qwk"]
}
rows.append(o)

o = {
    "method": "Linear Regression", 
    "weak_signal": "Z-score",
    "signal_filtering": "cmark",
    "text": "-",
    "ef": summary.loc[f"{our_model}/signal_clustering/w_sf/ef"]["qwk"],
    "nllf": summary.loc[f"{our_model}/signal_clustering/w_sf/nllf"]["qwk"],
    "ef+nllf": summary.loc[f"{our_model}/signal_clustering/w_sf/nllf_ef"]["qwk"]
}
rows.append(o)

o = {
    "method": "Linear Regression", 
    "weak_signal": "LLM-based signal",
    "signal_filtering": "cmark",
    "text": "-",
    "ef": summary.loc[f"{our_model}/llm/w_r_sf/ef"]["qwk"],
    "nllf": summary.loc[f"{our_model}/llm/w_r_sf/nllf"]["qwk"],
    "ef+nllf": summary.loc[f"{our_model}/llm/w_r_sf/nllf_ef"]["qwk"]
}
rows.append(o)

o = {
    "method": "Linear Regression", 
    "weak_signal": "ULRA-based signal",
    "signal_filtering": "cmark",
    "text": "-",
    "ef": summary.loc[f"{our_model}/ulra/w_sf/ef"]["qwk"],
    "nllf": summary.loc[f"{our_model}/ulra/w_sf/nllf"]["qwk"],
    "ef+nllf": summary.loc[f"{our_model}/ulra/w_sf/nllf_ef"]["qwk"]
}
rows.append(o)

o = {
    "method": "BERT", 
    "weak_signal": "Z-score",
    "signal_filtering": "xmark",
    "text": summary.loc[f"bert/signal_clustering/wo_sf"]["qwk"],
        "ef": "-",
    "nllf": "-",
    "ef+nllf": "-"
}
rows.append(o)

o = {
    "method": "BERT", 
    "weak_signal": "LLM-based signal",
    "signal_filtering": "xmark",
    "text": summary.loc[f"bert/llm/wo_sf"]["qwk"],
    "ef": "-",
    "nllf": "-",
    "ef+nllf": "-"
}
rows.append(o)

o = {
    "method": "BERT", 
    "weak_signal": "ULRA-based signal",
    "signal_filtering": "xmark",
    "text": summary.loc[f"bert/ulra/wo_sf"]["qwk"],
    "ef": "-",
    "nllf": "-",
    "ef+nllf": "-"
}
rows.append(o)

o = {
    "method": "BERT", 
    "weak_signal": "Z-score",
    "signal_filtering": "cmark",
    "text": summary.loc[f"bert/signal_clustering/w_sf"]["qwk"],
    "ef": "-",
    "nllf": "-",
    "ef+nllf": "-"
}
rows.append(o)

o = {
    "method": "BERT", 
    "weak_signal": "LLM-based signal",
    "signal_filtering": "cmark",
    "text": summary.loc[f"bert/llm/w_sf"]["qwk"],
    "ef": "-",
    "nllf": "-",
    "ef+nllf": "-"
}
rows.append(o)

o = {
    "method": "BERT", 
    "weak_signal": "ULRA-based signal",
    "signal_filtering": "cmark",
    "text": summary.loc[f"bert/ulra/w_sf"]["qwk"],
    "ef": "-",
    "nllf": "-",
    "ef+nllf": "-"
}
rows.append(o)

o = {
    "method": "human", 
    "weak_signal": "None",
    "signal_filtering": "-",
    "text": summary.loc[f"human"]["qwk"],
    "ef": "-",
    "nllf": "-",
    "ef+nllf": "-"
}
rows.append(o)

pd.DataFrame(rows)

,method,weak_signal,signal_filtering,text,ef,nllf,ef+nllf
0,Lenght,None,-,0.5186,-,-,-
1,Jaccard Sim.,None,-,-0.1566,-,-,-
2,Jaccard Sim.,None,-,-0.0036,-,-,-
3,ULRA,LF,-,0.6741,-,-,-
4,Z-score,None,-,0.7623,-,-,-
5,LLM,None,-,0.4858,-,-,-
6,LLM-CoT,None,-,0.2543,-,-,-
7,Linear Regression,Z-score,xmark,-,0.4722,0.5881,0.558
8,Linear Regression,LLM-based signal,xmark,-,0.4641,0.5546,0.6093
9,Linear Regression,ULRA-based signal,xmark,-,0.565,0.5655,0.6156


In [4]:
jx = 4

ulra_preds = scale_preds_per_set_and_model[jx]['ulra_paper/none/lf']#.keys()
our_preds = scale_preds_per_set_and_model[jx]["nllf_method/lr_Z_C/llm/w_r_sf/nllf_ef"]
essay_ids = essay_ids_per_set[jx]

comp = pd.DataFrame({
    "ulra_preds": ulra_preds,
    "our_preds": our_preds,
    "essay_ids": essay_ids
})

comp = comp.loc[(comp["ulra_preds"] - comp["our_preds"]).abs().sort_values().index]
comp[(comp["ulra_preds"]>= 1) & (comp["ulra_preds"]<= 6)].head(8)["essay_ids"].values

KeyError: 4